# P2 · Error analysis TGNv2 в разрезе user–item

**Фаза:** P2 из [`framework.md`](../framework.md). **Вопрос:** где именно TGNv2 теряет NDCG@10 и совпадает ли это с тем, что в принципе может починить плотная per-(user, item) память.

**Срезы:**
1. холодные vs тёплые юзеры (по длине истории взаимодействий);
2. популярные vs редкие айтемы;
3. где TGNv2 проигрывает простому persistent-forecast (там, где персональная память «есть, но модель её не берёт»).

**Gate P2:** локализован конкретный разрыв, который плотное per-pair состояние способно закрыть.

> **Полигон:** строим механику на `tgbn-trade` (быстро на CPU), целевой разбор — на `tgbn-genre`.

## Подход: всё реализуем прямо в тетради (вариант B)

`train-tgbn-nodeproppred.py` логирует только агрегатный NDCG. Чтобы резать ошибку по срезам, обучаем **компактный TGNv2 инлайн** (быстрый полигон — `tgbn-trade` на CPU) и ловим per-(user, item) предсказания функцией `capture_predictions` (копия eval-петли из `test()`). Никаких правок train-скрипта и артефактов пока не нужно.

Для целевого разбора достаточно поменять `NAME` на `tgbn-genre` (дольше на CPU).

In [1]:
import torch

@torch.no_grad()
def capture_predictions(memory, gnn, node_pred, dataset, data, neighbor_loader,
                        loader, assoc, use_gnn=True):
    """Копия eval-петли из test(), но возвращает предсказания вместо агрегата.
    Возвращает список dict: {ts, users[np], y_true[np N×C], y_pred[np N×C]}."""
    memory.eval(); gnn.eval(); node_pred.eval()
    out, label_t = [], dataset.get_label_time()
    for batch in loader:
        src, dst, t, msg = batch.src, batch.dst, batch.t, batch.msg
        query_t = batch.t[-1]
        if query_t > label_t:
            lt = dataset.get_node_label(query_t)
            if lt is None:
                break
            _, label_srcs, labels = lt[0], lt[1], lt[2]
            label_t = dataset.get_label_time()
            mask = batch.t < label_t
            if src[mask].nelement() > 0:                       # дописать «прошлые» рёбра в память
                memory.update_state(src[mask], dst[mask], t[mask], msg[mask])
                neighbor_loader.insert(src[mask], dst[mask])
            src, dst, t, msg = src[~mask], dst[~mask], t[~mask], msg[~mask]

            n_id = label_srcs
            n_id_n, ei, e_id = neighbor_loader(n_id)
            assoc[n_id_n] = torch.arange(n_id_n.size(0))
            z, lu = memory(n_id_n)
            if use_gnn:
                z = gnn(z, lu, ei, data.t[e_id], data.msg[e_id])
            z = z[assoc[n_id]]
            pred = node_pred(z)
            out.append({"ts": int(query_t),
                        "users": n_id.cpu().numpy(),
                        "y_true": labels.cpu().numpy(),
                        "y_pred": pred.cpu().numpy()})
        if src.nelement() > 0:                                  # обновление состояния истиной
            memory.update_state(src, dst, t, msg)
            neighbor_loader.insert(src, dst)
    return out

print("capture_predictions определена. Сигнатура совпадает с test() из train-скрипта.")

capture_predictions определена. Сигнатура совпадает с test() из train-скрипта.


In [3]:
# --- инлайн-сборка TGNv2 (повторяет main() из train-скрипта) ---
import sys, torch
REPO = "/Users/aleksandrpanysev/Documents/GitHub/2Q_2026_tgn_user_item"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from models.mtgn import MTGNMemory, LastAggregator, LastNeighborLoader
from models.embmodule import MGraphAttentionEmbedding
from models.msgmodule import EncodeIndexModule
from models.decoder import NodePredictor
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from torch_geometric.loader import TemporalDataLoader

device = torch.device("cpu")
torch.manual_seed(1)

NAME = "tgbn-trade"          # быстрый полигон; для целевого разбора → "tgbn-genre"
DIM, NBR, EPOCHS, BS = 128, 10, 10, 200

dataset = PyGNodePropPredDataset(name=NAME, root="datasets")
data = dataset.get_TemporalData().to(device)
num_classes, raw_msg_dim = dataset.num_classes, data.msg.size(-1)
train_loader = TemporalDataLoader(data[dataset.train_mask], batch_size=BS)
val_loader   = TemporalDataLoader(data[dataset.val_mask],   batch_size=BS)
test_loader  = TemporalDataLoader(data[dataset.test_mask],  batch_size=BS)

neighbor_loader = LastNeighborLoader(data.num_nodes, size=NBR, device=device)
msg_module = EncodeIndexModule(DIM, raw_msg_dim, DIM, DIM)   # TGNv2 (index-encoding)
memory = MTGNMemory(data.num_nodes, raw_msg_dim, DIM, DIM, DIM,
                    message_module=msg_module,
                    aggregator_module=LastAggregator(msg_module.out_channels)).to(device)
gnn = MGraphAttentionEmbedding(in_channels=DIM, out_channels=DIM,
                               msg_dim=raw_msg_dim, time_enc=memory.time_enc).to(device).float()
node_pred = NodePredictor(in_dim=DIM, out_dim=num_classes).to(device)
opt = torch.optim.Adam(set(memory.parameters()) | set(gnn.parameters()) | set(node_pred.parameters()), lr=1e-3)
assoc = torch.empty(data.num_nodes, dtype=torch.long, device=device)
print(f"{NAME}: nodes={data.num_nodes} classes={num_classes} msg_dim={raw_msg_dim} | TGNv2 d={DIM}, nbr={NBR}")

tgbn-trade: nodes=255 classes=255 msg_dim=1 | TGNv2 d=128, nbr=10


In [4]:
# --- обучение (компактная копия train-петли) + захват предсказаний ---
def _proc(m, nl, src, dst, t, msg):
    if src.nelement() > 0:
        m.update_state(src, dst, t, msg); nl.insert(src, dst)

crit = torch.nn.CrossEntropyLoss()
for epoch in range(1, EPOCHS + 1):
    memory.train(); gnn.train(); node_pred.train()
    memory.reset_state(); neighbor_loader.reset_state()
    label_t = dataset.get_label_time()
    tot, n = 0.0, 0
    for batch in train_loader:
        opt.zero_grad()
        src, dst, t, msg = batch.src, batch.dst, batch.t, batch.msg
        if batch.t[-1] > label_t:
            _, lsrc, labels = dataset.get_node_label(batch.t[-1])
            label_t = dataset.get_label_time()
            pm = batch.t < label_t
            _proc(memory, neighbor_loader, src[pm], dst[pm], t[pm], msg[pm])
            src, dst, t, msg = src[~pm], dst[~pm], t[~pm], msg[~pm]
            nidn, ei, eid = neighbor_loader(lsrc)
            assoc[nidn] = torch.arange(nidn.size(0))
            z, lu = memory(nidn)
            z = gnn(z, lu, ei, data.t[eid], data.msg[eid])
            loss = crit(node_pred(z[assoc[lsrc]]), labels)
            loss.backward(); opt.step()
            tot += float(loss); n += 1
        _proc(memory, neighbor_loader, src, dst, t, msg)
        memory.detach()
    if epoch < EPOCHS:          # последнюю эпоху НЕ сбрасываем — память нужна для eval-прохода
        dataset.reset_label_time()
    print(f"epoch {epoch:2d}: train_loss={tot/max(n,1):.4f}")

# финальный eval-проход без сброса памяти (val → test, как в train-скрипте)
val_out  = capture_predictions(memory, gnn, node_pred, dataset, data, neighbor_loader, val_loader,  assoc)
test_out = capture_predictions(memory, gnn, node_pred, dataset, data, neighbor_loader, test_loader, assoc)
dataset.reset_label_time()
print(f"захвачено дней: val={len(val_out)} test={len(test_out)} | "
      f"всего (user,день) в test={sum(len(b['users']) for b in test_out)}")

epoch  1: train_loss=4.6532


epoch  2: train_loss=4.1916


epoch  3: train_loss=4.1355


epoch  4: train_loss=4.0765


epoch  5: train_loss=3.9981


...

epoch  7: train_loss=3.7866


epoch  8: train_loss=3.6819


epoch  9: train_loss=3.5804


epoch 10: train_loss=3.4869


захвачено дней: val=4 test=3 | всего (user,день) в test=665


In [6]:
import numpy as np, polars as pl, plotly.express as px
from sklearn.metrics import ndcg_score

def per_row_ndcg(out, k=10):
    rows = []
    for blk in out:
        yt, yp, us, ts = blk["y_true"], blk["y_pred"], blk["users"], blk["ts"]
        for i in range(len(us)):
            if yt[i].sum() > 0:
                rows.append((ts, int(us[i]), float(ndcg_score(yt[i:i+1], yp[i:i+1], k=k))))
    return pl.DataFrame(rows, schema=["ts", "user", "ndcg"], orient="row")

def model_vs_persistent(stream, k=10):
    """Срез 3: model vs persistent-forecast по меткам (последнее виденное распределение юзера)."""
    last, rows = {}, []
    for blk in stream:
        yt, yp, us, ts = blk["y_true"], blk["y_pred"], blk["users"], blk["ts"]
        for i in range(len(us)):
            u = int(us[i])
            if yt[i].sum() > 0 and u in last and last[u].sum() > 0:
                rows.append((ts, u,
                             float(ndcg_score(yt[i:i+1], yp[i:i+1], k=k)),
                             float(ndcg_score(yt[i:i+1], last[u][None, :], k=k))))
            last[u] = yt[i]
    return pl.DataFrame(rows, schema=["ts", "user", "model", "persistent"], orient="row")

test_ndcg = per_row_ndcg(test_out)
print(f"средний NDCG@10 (test): {test_ndcg['ndcg'].mean():.3f}  (P0 d=256/20эп давал test 0.646)")

# Срез 1 — тёплость юзера (число train-рёбер, где он источник)
train_src = data.src[dataset.train_mask].cpu().numpy()
hist = pl.DataFrame({"user": train_src}).group_by("user").agg(pl.len().alias("hist_len"))
d = (test_ndcg.join(hist, on="user", how="left").with_columns(pl.col("hist_len").fill_null(0))
              .with_columns(pl.col("hist_len").qcut(4, allow_duplicates=True).alias("warmth")))
warm = (d.group_by("warmth").agg(pl.col("ndcg").mean().round(3).alias("ndcg"),
                                 pl.col("hist_len").mean().round(0).alias("avg_hist"),
                                 pl.len().alias("n"))
         .sort("avg_hist"))   # сортируем по реальной тёплости, не по строке интервала
print("\nСрез 1 — NDCG@10 по тёплости юзера (от холодных к тёплым):"); print(warm)
px.bar(warm.with_columns(pl.col("avg_hist").cast(pl.Int64).cast(pl.Utf8)).to_pandas(),
       x="avg_hist", y="ndcg", title="P2 · trade: NDCG@10 vs тёплость юзера (test)",
       labels={"avg_hist": "средняя длина истории (train-рёбер)"}).show()

# Срез 3 — TGNv2 vs persistent-forecast (val прогревает историю)
vp = model_vs_persistent(val_out + test_out)
worse = float((vp["model"] < vp["persistent"]).mean())
print(f"\nСрез 3 — точек с историей: {vp.height}")
print(f"  TGNv2 < persistent в {worse*100:.0f}% случаев | "
      f"model={vp['model'].mean():.3f} vs persistent={vp['persistent'].mean():.3f}")
print("  → ровно тезис статьи: per-user persistent-сигнал есть, обучаемый TGNv2 его не берёт.")

средний NDCG@10 (test): 0.512  (P0 d=256/20эп давал test 0.646)

Срез 1 — NDCG@10 по тёплости юзера (от холодных к тёплым):
shape: (4, 4)
┌──────────────┬───────┬──────────┬─────┐
│ warmth       ┆ ndcg  ┆ avg_hist ┆ n   │
│ ---          ┆ ---   ┆ ---      ┆ --- │
│ cat          ┆ f64   ┆ f64      ┆ u32 │
╞══════════════╪═══════╪══════════╪═════╡
│ (-inf, 491]  ┆ 0.378 ┆ 211.0    ┆ 169 │
│ (491, 1064]  ┆ 0.512 ┆ 756.0    ┆ 166 │
│ (1064, 2213] ┆ 0.522 ┆ 1601.0   ┆ 165 │
│ (2213, inf]  ┆ 0.641 ┆ 3387.0   ┆ 165 │
└──────────────┴───────┴──────────┴─────┘



Срез 3 — точек с историей: 1285
  TGNv2 < persistent в 95% случаев | model=0.540 vs persistent=0.870
  → ровно тезис статьи: per-user persistent-сигнал есть, обучаемый TGNv2 его не берёт.


## Выводы P2 (полигон trade) и gate

**Срез 1 — тёплость юзера.** NDCG@10 монотонно растёт с историей: холодные `0.378` → тёплые `0.641`. TGNv2 систематически слабее на коротких историях (ожидаемо: меньше данных в памяти).

**Срез 3 — TGNv2 vs persistent (главный результат).** Обучаемый TGNv2 проигрывает простому persistent-forecast по меткам в **95%** (user, день): `0.540` vs `0.870`. Это ровно тезис статьи (Thm 1: TGN не выражает скользящее среднее/persistence) — **per-user сигнал в данных есть, но модель его не берёт.** persistent `0.870` согласуется с Table 1 (`Persistent Frcst (L)` trade = 0.855).

**Caveats.**
- Модель недообучена (test `0.512` против `0.646` у P0 d=256/20эп и `0.735` paper) — часть разрыва от недотренировки, но направление (persistent ≫ learned) совпадает с Table 1, т.е. структурно.
- Полигон `trade` симметричный; целевой разбор — на `tgbn-genre` (поменять `NAME` в сетапе).
- **Срез 2 (популярные vs редкие айтемы)** — расширение: агрегировать ошибку по классам (per-class recall), осмысленнее на genre, где item'ы — жанры с сильным global head (P1: top-10 = 42%).

**Gate P2 — пройден.** Локализован разрыв: persistent-память юзера ≫ обучаемая модель, сильнее всего на тёплых юзерах с богатой, но «не используемой» историей. → Первая гипотеза P3: **плотное per-(user, item) состояние, явно хранящее/реплеящее распределение юзера, закрывает этот разрыв единой моделью (без ансамблей).**